# 📊 Báo Cáo Phân Tích Dữ Liệu EV Stations (Synthetic Data)

## 1. Đánh Giá Bản Chất Bộ Dữ Liệu
Bộ dữ liệu gồm **1.5 triệu dòng** này mang tính chất **Synthetic (Dữ liệu giả lập)** phục vụ mục đích học tập và kiểm thử hiệu năng (Stress Test). Qua quan sát, dữ liệu có các đặc điểm "nhiễu" sau:

* **Tọa độ ngẫu nhiên:** Kinh độ và vĩ độ được rải ngẫu nhiên trên toàn thế giới theo hệ tọa độ $WGS84$. Điều này khiến phần lớn các điểm rơi vào đại dương hoặc các vùng hoang mạc không có dân cư.
* **Thiếu tính nhất quán:** Tên quốc gia (Japan, Norway, China...) và thành phố không khớp với tọa độ thực tế. Ví dụ: Một trạm sạc có địa chỉ tại Nhật Bản nhưng tọa độ lại nằm ở Nam Thái Bình Dương.
* **Mục tiêu sử dụng:** Bộ dữ liệu này không dùng để tra cứu vị trí thực tế mà dùng để thực hành kỹ năng:
    * Xử lý tập dữ liệu lớn (**Big Data**).
    * Tối ưu hóa render bản đồ (**Marker Cluster**).
    * Thực hiện các phép toán hình học không gian.

---

## 2. Logic "Clone" Dữ Liệu Về TP. Hồ Chí Minh
Vì dữ liệu gốc bị phân tán quá xa, chúng ta sẽ áp dụng logic **Spatial Re-mapping** (Tái cấu trúc không gian) để đưa dữ liệu về một khu vực có ý nghĩa phân tích hơn (TP.HCM).

### Các bước thực hiện:
1.  **Lấy mẫu ngẫu nhiên (Random Sampling):** Do trình duyệt không thể hiển thị cùng lúc 1.5 triệu điểm, ta lấy mẫu tỷ lệ $1/1000$ (tương đương khoảng 1,500 trạm).
2.  **Định vị khu vực đích (Target Bounding Box):** Xác định khung tọa độ bao quanh TP.HCM:
    * Vĩ độ ($Lat$): $[10.70, 10.90]$
    * Kinh độ ($Lon$): $[106.60, 106.80]$
3.  **Tái tạo tọa độ:** Sử dụng hàm phân phối ngẫu nhiên để ép các trạm sạc vào trong khung tọa độ của TP.HCM. Việc này giúp biến các "trạm sạc toàn cầu" thành các "trạm sạc tiềm năng" tại địa phương.
4.  **Địa phương hóa dữ liệu:** Cập nhật lại cột `City` thành "Ho Chi Minh City" và `Country` thành "Vietnam" để đồng bộ hóa báo cáo.

---

## 3. Ý Nghĩa Đối Với Bài Toán Green Logistics
Việc đưa dữ liệu về một khu vực cụ thể giúp chúng ta giải quyết các bài toán logic thực tế:

* **Phân tích sự giao thoa:** So sánh vị trí các trạm sạc xe điện với các **Cây xăng hiện hữu** (Petrolimex, Comeco...) để đánh giá tốc độ chuyển đổi năng lượng xanh.
* **Tối ưu hóa vị trí:** Sử dụng các trạm sạc đã clone làm dữ liệu đầu vào để tính toán khoảng cách bao phủ (Coverage) trong khu vực nội thành.
* **Xây dựng Heatmap:** Xác định các khu vực có mật độ trạm sạc dày đặc hoặc các "vùng trắng" cần đầu tư thêm.

In [2]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd

DATA_PATH = "/content/drive/MyDrive/Colab Notebooks/datastorm_round_2_green_logistics/data/01_raw/ev_stations.csv"
df = pd.read_csv(DATA_PATH)

print("Rows:", df.shape[0])
df.head()


Mounted at /content/drive
Rows: 1500000


,id,country,city,network,latitude,longitude,connectors_available,connector_types,power_level,max_power_kw,is_24_7,cost_per_kwh_usd,status,install_year,accessibility
0,1,United States,Castilloshire,Ionity,14.540228,-78.268332,4,"CCS, Tesla NACS, Type 2",Level 1 (Slow),22,True,0.10,Operational,2025,Public
1,2,Switzerland,Hannahfurt,ChargePoint,4.854897,-14.876427,2,CHAdeMO,Level 2 (Medium),22,True,0.44,Operational,2022,Public
2,3,Japan,Dylanhaven,Tesla Supercharger,-26.813349,-144.282421,3,Tesla NACS,Ultra-Fast (>150kW),50,True,0.33,Under Maintenance,2025,Private
3,4,Norway,Thomasborough,Ionity,-1.221722,-19.138671,6,Tesla NACS,Ultra-Fast (>150kW),50,False,0.23,Operational,2024,Public
4,5,China,合山县,EVgo,-24.282606,-106.890714,4,"GB/T, CCS, J1772",DC Fast (>50kW),22,True,0.18,Operational,2017,Public


In [ ]:
station_df = (
    df
    .groupby(['latitude', 'longitude'])
    .agg({
        'network': lambda x: ','.join(set(x)),
        'connector_types': lambda x: ','.join(set(x)),
        'max_power_kw': 'max'
    })
    .reset_index()
)

print("Stations:", station_df.shape[0])
station_df.head()


Stations: 1500000


,latitude,longitude,network,connector_types,max_power_kw
0,-89.999929,50.353415,EVgo,Type 2,50
1,-89.999851,7.903471,EVgo,GB/T,50
2,-89.999821,-99.574914,Ionity,Tesla NACS,50
3,-89.999740,85.142694,Star Charge (China),J1772,22
4,-89.999647,118.351115,Tesla Supercharger,"GB/T, Tesla NACS",22


In [3]:
import folium
from folium.plugins import MarkerCluster

# 1. Lấy mẫu ngẫu nhiên 1/1000 dữ liệu
# frac=0.001 tương đương 1/1000 (khoảng 1500 trạm từ 1.5tr dòng)
df_sample = df.sample(frac=0.001, random_state=42)

print(f"Số lượng trạm sau khi lấy mẫu: {len(df_sample)}")

# 2. Khởi tạo bản đồ (tự động lấy trung tâm dựa trên dữ liệu mẫu)
avg_lat = df_sample['latitude'].mean()
avg_lon = df_sample['longitude'].mean()
m = folium.Map(location=[avg_lat, avg_lon], zoom_start=2, tiles="cartodbpositron")

# 3. Sử dụng MarkerCluster để bản đồ mượt mà hơn khi zoom xa
marker_cluster = MarkerCluster().add_to(m)

# 4. Thêm các trạm vào cluster
for _, row in df_sample.iterrows():
    # Tạo nội dung popup
    popup_text = f"""
    <b>Network:</b> {row['network']}<br>
    <b>Power:</b> {row['max_power_kw']} kW<br>
    <b>Connectors:</b> {row['connector_types']}<br>
    <b>Status:</b> {row['status']}
    """

    folium.Marker(
        location=[row['latitude'], row['longitude']],
        popup=folium.Popup(popup_text, max_width=300),
        icon=folium.Icon(color="blue", icon="bolt", prefix="fa")
    ).add_to(marker_cluster)

# Hiển thị bản đồ
m

Số lượng trạm sau khi lấy mẫu: 1500


In [ ]:
lat_min, lat_max = 10.5, 11.2
lon_min, lon_max = 106.4, 107.0

hcmc_df = station_df[
    (station_df.latitude >= lat_min) &
    (station_df.latitude <= lat_max) &
    (station_df.longitude >= lon_min) &
    (station_df.longitude <= lon_max)
]

print("Filtered stations near HCMC:", hcmc_df.shape[0])
hcmc_df.head()


Filtered stations near HCMC: 10


,latitude,longitude,network,connector_types,max_power_kw
837619,10.555018,106.939900,BP Pulse,"J1772, Tesla NACS",150
838036,10.607752,106.613831,EVgo,GB/T,50
839292,10.753683,106.826587,Tesla Supercharger,CCS,250
839473,10.775825,106.542119,EVgo,"CCS, GB/T, J1772",50
839712,10.804199,106.630715,Blink Charging,"J1772, CCS",150


In [ ]:
import folium

# Center map on HCMC
hcmc_center = [(lat_min + lat_max) / 2, (lon_min + lon_max) / 2]
m = folium.Map(location=hcmc_center, zoom_start=12, tiles="cartodbpositron")m


In [ ]:
for _, row in hcmc_df.iterrows():
    folium.CircleMarker(
        location=[row.latitude, row.longitude],
        radius=4,
        color="blue",
        fill=True,
        fill_opacity=0.7,
        popup=f"Power: {row.max_power_kw} kW\nNetwork: {row.network}"
    ).add_to(m)

m


In [ ]:
from folium.plugins import HeatMap

heat_data = hcmc_df[['latitude','longitude']].values.tolist()

HeatMap(heat_data, radius=15, blur=25, min_opacity=0.4).add_to(m)
m


----

In [ ]:
lat_min, lat_max = 10.0, 12.0
lon_min, lon_max = 105.5, 107.5

hcmc_df = station_df[
    (station_df.latitude >= lat_min) &
    (station_df.latitude <= lat_max) &
    (station_df.longitude >= lon_min) &
    (station_df.longitude <= lon_max)
]

print("Stations in expanded region:", hcmc_df.shape[0])


Stations in expanded region: 71


In [ ]:
from folium.plugins import MarkerCluster

marker_cluster = MarkerCluster().add_to(m)

for _, row in hcmc_df.iterrows():
    folium.Marker(
        location=[row.latitude, row.longitude],
        icon=None,
        popup=f"{row.network} — {row.max_power_kw}kW"
    ).add_to(marker_cluster)

m


cây xăng

In [ ]:
import pandas as pd

gas_stations = [
    {"name":"Petrolimex No.03","lat":10.7712,"lon":106.7023},
    {"name":"Petrolimex No.01","lat":10.7754,"lon":106.7011},
    {"name":"Petrolimex No.38","lat":10.7927,"lon":106.6892},
    {"name":"Petrolimex No.21","lat":10.8105,"lon":106.7092},
    {"name":"Comeco 3","lat":10.8078,"lon":106.7221},
    {"name":"Petrolimex No.06","lat":10.7836,"lon":106.6881},
    {"name":"Satra No.18","lat":10.7760,"lon":106.6941},
    {"name":"Petrolimex No.30","lat":10.8234,"lon":106.6987},
    {"name":"STS No.13","lat":10.7803,"lon":106.6905},
    {"name":"Hiếu Phương 3","lat":10.8067,"lon":106.7333},
    {"name":"Petrolimex No.53","lat":10.7929,"lon":106.6852},
    {"name":"Petrolimex No.39","lat":10.7570,"lon":106.6885},
    {"name":"Petrolimex No.19","lat":10.8061,"lon":106.7221},
    {"name":"Sasco","lat":10.7875,"lon":106.6540},
    {"name":"PVOIL Comeco","lat":10.7609,"lon":106.6580},
    {"name":"Petrolimex No.11","lat":10.7633,"lon":106.6548},
]

df_gas = pd.DataFrame(gas_stations)


In [ ]:
import folium

# Tâm map TP HCM
center = [10.7769, 106.70098]
m = folium.Map(location=center, zoom_start=12, tiles="cartodbpositron")

# Thêm marker cây xăng
for _, row in df_gas.iterrows():
    folium.Marker(
        location=[row.lat, row.lon],
        popup=row.name,
        icon=folium.Icon(color="red", icon="oil", prefix='fa')
    ).add_to(m)

m


In [ ]:
from folium.plugins import MarkerCluster

cluster = MarkerCluster().add_to(m)

for _, row in df_gas.iterrows():
    folium.Marker(
        location=[row.lat, row.lon],
        popup=row.name
    ).add_to(cluster)

m


In [ ]:
from folium.plugins import HeatMap

heat_data = df_gas[['lat','lon']].values.tolist()
HeatMap(heat_data, radius=15).add_to(m)
m
